# Explicabilidad del modelo final

Laboratorio 10 de CC3092. Explica con Grad-CAM el modelo final del proyecto (`mnv3_ft_cost`,
MobileNetV3-Small, semilla 42) sobre el conjunto de test, sin reentrenar nada. Como alternativa se
usa oclusión, y como baseline interpretable el modelo B1 (HOG + regresión logística).

Qué genera (todo en `results/xai/`):

| Archivo | Contenido |
|---|---|
| `parametros.json` | parámetros de diseño de cada explicación |
| `instancias.csv` | las instancias locales y por qué se eligieron |
| `locales.png` | imagen, Grad-CAM y oclusión de cada instancia local |
| `global.png` y `global.csv` | mapa promedio por clase y clima, y concentración en el centro |
| `fidelidad.csv` | caída del margen al tapar la zona que marca Grad-CAM, frente al azar |
| `sensibilidad.csv` | qué tanto cambian los mapas y la fidelidad al cambiar la capa o la ventana |
| `errores_ocupado_libre.png` | Grad-CAM de los errores ocupado→libre de test |
| `baseline_b1.png` | contribución por bloque HOG del modelo B1 contra Grad-CAM |
| `costo.csv` | milisegundos por explicación en GPU y en CPU con un hilo |

Requisitos: el run `runs/mnv3_ft_cost_s42_*` (se obtiene entrenando `configs/mnv3_ft_cost.yaml` con
`--seed 42`), el run de B1 (`configs/b1_hog_lr.yaml`) y el dataset en `DATA_ROOT`. Todo usa la semilla 42.

## 1. Configuración

In [1]:
import copy
import glob
import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]

import gradcam as G          # Grad-CAM del proyecto (scripts/gradcam.py)
from cupo import explain as X
from cupo.runs import load_run

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

OUT = ROOT / "results" / "xai"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
print("dispositivo:", DEVICE)

dispositivo: cuda


## 2. Modelo y punto de operación

Se carga el run final tal como quedó entrenado. Los umbrales t_L y t_H y la temperatura son los que
se eligieron con validación. En test solo se explican predicciones, no se elige nada.

In [2]:
run_dirs = sorted(glob.glob(str(ROOT / "runs" / "mnv3_ft_cost_s42_*")))
assert run_dirs, "No está el run mnv3_ft_cost_s42_*; ver README para generarlo"
run = load_run(run_dirs[-1], DEVICE)
G.disable_inplace(run.model)
arch = run.config["model"]["name"]
op = X.operating_point(run)
print("run:", run.run_dir.name, "| arquitectura:", arch, "| entrada:", run.input_size)
print(f"escenario {op.scenario}: t_L = {op.t_L}, t_H = {op.t_H}, temperatura = {op.temperature}")

test = X.load_split_table(run, "test")
test["decision"] = X.decision(test.p.to_numpy(), op.t_L, op.t_H)
print(len(test), "imágenes de test")
test.groupby(["y", "decision"]).size().unstack(fill_value=0)

run: mnv3_ft_cost_s42_20261112-1505 | arquitectura: mobilenet_v3_small | entrada: 224
escenario comercial: t_L = 0.01, t_H = 0.23, temperatura = 1.0
31825 imágenes de test


decision,libre,ocupado,por verificar
y,,,
0,12581,62,906
1,10,18242,24


## 3. Parámetros de diseño

Cada valor afecta la explicación. Se dejan fijos y quedan guardados en `parametros.json`.

In [3]:
PATCH, STRIDE, FRACTION, N_RANDOM, PER_STRATUM = 56, 28, 0.2, 5, 84
params = {
    "semilla": SEED,
    "run": run.run_dir.name,
    "capa_objetivo_gradcam": "model.features[-1] (última Conv2dNormActivation de MobileNetV3-Small)",
    "puntaje_explicado": "margen z_clase - z_otra de la clase de la decisión (ocupado si p >= t_H, libre si p < t_L, y en por verificar la más probable)",
    "oclusion_ventana_px": PATCH,
    "oclusion_paso_px": STRIDE,
    "oclusion_relleno": "0 en la entrada normalizada (color medio de ImageNet)",
    "fidelidad_fraccion_tapada": FRACTION,
    "fidelidad_cuadrados_al_azar": N_RANDOM,
    "global_imagenes_por_clase_y_clima": PER_STRATUM,
    "global_centro_lado_relativo": 0.5,
    "umbrales": {"t_L": op.t_L, "t_H": op.t_H, "temperatura": op.temperature},
}
(OUT / "parametros.json").write_text(json.dumps(params, indent=2, ensure_ascii=False), encoding="utf-8")
pd.Series(params).to_frame("valor")

,valor
semilla,42
run,mnv3_ft_cost_s42_20261112-1505
capa_objetivo_gradcam,model.features[-1] (última Conv2dNormActivatio...
puntaje_explicado,margen z_clase - z_otra de la clase de la deci...
oclusion_ventana_px,56
oclusion_paso_px,28
oclusion_relleno,0 en la entrada normalizada (color medio de Im...
fidelidad_fraccion_tapada,0.2
fidelidad_cuadrados_al_azar,5
global_imagenes_por_clase_y_clima,84


## 4. Explicaciones locales

Seis instancias de test elegidas con reglas fijas (`explain.select_instances`): un acierto de ocupado y
uno de libre con confianza alta, un error ocupado→libre, el caso más cercano a t_H, un acierto con
lluvia (subgrupo de interés, porque la validación no tiene lluvia) y un error libre→ocupado.

In [4]:
sel = X.select_instances(test, op.t_L, op.t_H, SEED)
sel_csv = sel.assign(path=sel["path"].map(G.data_relative))
sel_csv.to_csv(OUT / "instancias.csv", index=False)
sel.drop(columns=["path"])

,motivo,row,y,p,decision,camera,weather,date,time,slot_id
0,acierto ocupado (confianza alta),2906,1,1.000000,ocupado,9,SUNNY,2015-12-17,16:54,255
1,acierto libre (confianza alta),24659,0,0.000132,libre,8,SUNNY,2016-01-16,13:21,282
2,error ocupado→libre,25941,1,0.000279,libre,5,RAINY,2015-12-22,14:23,236
3,cerca del umbral t_H,3006,0,0.223292,por verificar,7,RAINY,2016-02-12,08:55,210
4,acierto con lluvia,20887,1,1.000000,ocupado,4,RAINY,2015-12-22,11:21,195
5,error libre→ocupado,18202,0,1.000000,ocupado,7,SUNNY,2015-12-17,07:24,208


In [5]:
cam = G.GradCAM(run.model, G.target_layer(run.model, arch))


def explain_batch(df):
    """Imágenes, mapas Grad-CAM y clase explicada (la de la decisión) de las filas de ``df``."""
    batch, shown = G.load_images(df["path"].tolist(), run.input_size)
    x = torch.from_numpy(batch).to(DEVICE)
    target = G.decision_class(df["p"].to_numpy(), op.t_L, op.t_H)
    cams = np.concatenate([cam(x[i:i + 64], target[i:i + 64])[0] for i in range(0, len(x), 64)])
    return x, shown, cams, target


def overlay(ax, img, heat, title=None):
    ax.imshow(img)
    if heat is not None:
        ax.imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=8)


x_sel, shown_sel, cams_sel, target_sel = explain_batch(sel)
occ_sel = [X.normalize_map(X.occlusion_map(run.model, x_sel[i:i + 1], int(target_sel[i]), PATCH, STRIDE))
           for i in range(len(sel))]

NAMES = {0: "libre", 1: "ocupado"}
fig, axes = plt.subplots(len(sel), 3, figsize=(7.5, 2.55 * len(sel)))
for i, r in sel.iterrows():
    head = f"{i + 1}. {r.motivo}\ny={NAMES[r.y]}  p={r.p:.4f}  cámara {r.camera}  {r.weather}"
    overlay(axes[i, 0], shown_sel[i], None, head)
    overlay(axes[i, 1], shown_sel[i], cams_sel[i], f"Grad-CAM (explica {NAMES[target_sel[i]]})")
    overlay(axes[i, 2], shown_sel[i], occ_sel[i], "Oclusión")
fig.tight_layout()
fig.savefig(OUT / "locales.png", dpi=130)
plt.close(fig)

![locales.png](../results/xai/locales.png)

## 5. Explicación global

Muestra de test con el mismo número de imágenes por clase real y clima (semilla 42). Para cada
imagen se calcula el Grad-CAM de la clase de la decisión. Se muestra el mapa promedio de cada grupo y
la parte de la atención que cae en el cuadrado central del parche (un cuadrado de la mitad del lado
cubre el 25 % del área, así que 0.25 sería atención pareja).

In [6]:
sample = X.stratified_sample(test, PER_STRATUM, SEED)
x_s, shown_s, cams_s, target_s = explain_batch(sample)
sample["centro"] = X.center_share(cams_s)
print(len(sample), "imágenes")

groups = [(y, w) for y in (0, 1) for w in ("SUNNY", "OVERCAST", "RAINY")]
fig, axes = plt.subplots(2, 3, figsize=(8, 5.6))
rows = []
for ax, (y, w) in zip(axes.ravel(), groups):
    idx = np.where((sample.y == y) & (sample.weather == w))[0]
    mean_map = cams_s[idx].mean(axis=0)
    ax.imshow(mean_map, cmap="jet", vmin=0, vmax=1)
    ax.set_title(f"{NAMES[y]} real, {w}\nn={len(idx)}  centro={sample.centro.values[idx].mean():.2f}", fontsize=8)
    ax.axis("off")
    rows.append({"y": NAMES[y], "weather": w, "n": len(idx),
                 "centro_medio": sample.centro.values[idx].mean(),
                 "centro_std": sample.centro.values[idx].std(ddof=1)})
fig.tight_layout()
fig.savefig(OUT / "global.png", dpi=130)
plt.close(fig)
glob_df = pd.DataFrame(rows)
glob_df.to_csv(OUT / "global.csv", index=False)
glob_df.round(3)

504 imágenes


,y,weather,n,centro_medio,centro_std
0,libre,SUNNY,84,0.192,0.060
1,libre,OVERCAST,84,0.208,0.066
2,libre,RAINY,84,0.211,0.076
3,ocupado,SUNNY,84,0.404,0.116
4,ocupado,OVERCAST,84,0.427,0.111
5,ocupado,RAINY,84,0.420,0.110


![global.png](../results/xai/global.png)

### Fidelidad: ¿Grad-CAM marca zonas que de verdad importan?

Por cada imagen se tapa el 20 % de los píxeles (relleno medio) y se mide cuánto baja el margen de la
clase explicada. Se compara tapar las zonas de mayor valor de Grad-CAM contra tapar las de menor valor
y contra un cuadrado del mismo tamaño en posición al azar (5 por imagen). También se mide qué tanto se
parece el mapa de Grad-CAM al de oclusión (correlación de Pearson de los dos mapas).

In [7]:
fid = X.deletion_fidelity(run.model, x_s, cams_s, target_s, FRACTION, N_RANDOM, seed=SEED)
fid = pd.concat([sample[["y", "weather"]].reset_index(drop=True), fid], axis=1)
fid["top_mayor_que_azar"] = fid.caida_top > fid.caida_azar

occ_s = np.stack([X.normalize_map(X.occlusion_map(run.model, x_s[i:i + 1], int(target_s[i]), PATCH, STRIDE))
                  for i in range(len(sample))])
fid["corr_gradcam_oclusion"] = [np.corrcoef(cams_s[i].ravel(), occ_s[i].ravel())[0, 1] if occ_s[i].std() > 0 else np.nan
                                for i in range(len(sample))]


def boot_ci(v, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, float)
    means = [rng.choice(v, size=len(v), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])


def summarize(g):
    lo, hi = boot_ci(g.caida_top - g.caida_azar)
    return pd.Series({"n": len(g), "caida_top": g.caida_top.mean(), "caida_azar": g.caida_azar.mean(),
                      "caida_bottom": g.caida_bottom.mean(),
                      "top_menos_azar": (g.caida_top - g.caida_azar).mean(),
                      "ic95_lo": lo, "ic95_hi": hi,
                      "pct_top_mayor_azar": 100 * g.top_mayor_que_azar.mean(),
                      "corr_gradcam_oclusion": g.corr_gradcam_oclusion.mean()})


fid_table = pd.concat([summarize(fid).to_frame("todas").T,
                       fid.groupby("weather").apply(summarize, include_groups=False)])
fid_table.to_csv(OUT / "fidelidad.csv")
fid_table.round(3)

,n,caida_top,caida_azar,caida_bottom,top_menos_azar,ic95_lo,ic95_hi,pct_top_mayor_azar,corr_gradcam_oclusion
todas,504.0,3.966,1.007,-0.644,2.959,2.713,3.228,93.056,0.308
OVERCAST,168.0,3.544,0.811,-0.780,2.734,2.287,3.197,92.857,0.323
RAINY,168.0,4.389,1.328,-0.453,3.061,2.586,3.547,92.857,0.321
SUNNY,168.0,3.963,0.882,-0.700,3.081,2.643,3.536,93.452,0.281


## 6. Errores ocupado→libre de test

El error más caro del sistema es mostrar libre un espacio que está ocupado. Estos son todos los que
hay en test con los umbrales finales, con el Grad-CAM de la clase que decidió el modelo (libre).

In [8]:
errs = test[(test.y == 1) & (test.decision == "libre")].sort_values("p").reset_index().rename(columns={"index": "row"})
x_e, shown_e, cams_e, target_e = explain_batch(errs)
cols = 5
fig, axes = plt.subplots(int(np.ceil(len(errs) / cols)), cols, figsize=(2.3 * cols, 2.6 * np.ceil(len(errs) / cols)))
for ax in axes.ravel():
    ax.axis("off")
for i, r in errs.iterrows():
    overlay(axes.ravel()[i], shown_e[i], cams_e[i], f"p={r.p:.4f} cam {r.camera}\n{r.date} {r.time}")
fig.tight_layout()
fig.savefig(OUT / "errores_ocupado_libre.png", dpi=130)
plt.close(fig)
errs.drop(columns=["path"]).assign(centro=X.center_share(cams_e)).round(4)

,row,y,p,camera,weather,date,time,slot_id,decision,centro
0,25941,1,0.0003,5,RAINY,2015-12-22,14:23,236,libre,0.2067
1,31152,1,0.0006,8,RAINY,2016-02-12,08:01,321,libre,0.2233
2,5552,1,0.0007,8,RAINY,2016-02-12,08:01,320,libre,0.2904
3,25985,1,0.0008,8,RAINY,2016-02-12,07:30,320,libre,0.2708
4,15934,1,0.0030,4,RAINY,2015-12-22,09:21,225,libre,0.2265
5,11858,1,0.0033,7,RAINY,2016-02-12,09:55,612,libre,0.2402
6,8437,1,0.0044,7,RAINY,2016-02-12,17:25,612,libre,0.3120
7,10677,1,0.0057,9,RAINY,2016-02-12,17:42,258,libre,0.2836
8,22880,1,0.0087,8,RAINY,2016-02-12,07:30,321,libre,0.2790
9,26377,1,0.0093,7,RAINY,2016-02-12,10:55,612,libre,0.2067


![errores_ocupado_libre.png](../results/xai/errores_ocupado_libre.png)

## 7. Baseline interpretable: HOG + regresión logística (B1)

B1 es lineal, así que su explicación es exacta: cada bloque HOG aporta `w · z` al log-odds de ocupado.
Se muestra ese aporte con signo (rojo empuja a ocupado, azul a libre) junto al Grad-CAM de las mismas
seis instancias.

In [9]:
b1_dirs = sorted(glob.glob(str(ROOT / "runs" / "b1_hog_lr_s42_*")))
assert b1_dirs, "No está el run b1_hog_lr_s42_*; ver README para generarlo"
b1 = joblib.load(Path(b1_dirs[-1]) / "model.joblib")

fig, axes = plt.subplots(len(sel), 3, figsize=(7.5, 2.4 * len(sel)))
for i, r in sel.iterrows():
    contrib = X.hog_contribution_map(b1, r.path)
    lim = max(np.abs(contrib).max(), 1e-6)
    overlay(axes[i, 0], shown_sel[i], None, f"{i + 1}. y={NAMES[r.y]}  p modelo final={r.p:.3f}")
    axes[i, 1].imshow(contrib, cmap="coolwarm", vmin=-lim, vmax=lim)
    axes[i, 1].set_title(f"B1: aporte por bloque (suma {contrib.sum():+.1f})", fontsize=8)
    axes[i, 1].axis("off")
    overlay(axes[i, 2], shown_sel[i], cams_sel[i], "Grad-CAM")
fig.tight_layout()
fig.savefig(OUT / "baseline_b1.png", dpi=130)
plt.close(fig)

![baseline_b1.png](../results/xai/baseline_b1.png)

## 8. Costo computacional

Milisegundos por explicación de una imagen (mediana de 30 repeticiones tras calentar). La predicción
sola es el punto de comparación. En CPU se usa un hilo, como en el benchmark del proyecto.
Los tiempos cambian de una corrida a otra y de una máquina a otra. Todo lo demás del notebook da
exactamente el mismo resultado en cada corrida con la semilla 42.

In [10]:
def median_ms(fn, reps=30, sync=False):
    fn()
    times = []
    for _ in range(reps):
        if sync and DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync and DEVICE == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def timings(model, layer, x1, tgt, sync):
    cam_local = G.GradCAM(model, layer)
    out = {
        "prediccion": median_ms(lambda: model(x1), sync=sync),
        "gradcam": median_ms(lambda: cam_local(x1, [tgt]), sync=sync),
        "oclusion": median_ms(lambda: X.occlusion_map(model, x1, tgt, PATCH, STRIDE), sync=sync),
    }
    cam_local.close()
    return out


rows = []
with torch.no_grad():
    rows.append({"dispositivo": DEVICE, **timings(run.model, G.target_layer(run.model, arch), x_sel[:1], int(target_sel[0]), True)})
cpu_model = copy.deepcopy(run.model).cpu().eval()
torch.set_num_threads(1)
rows.append({"dispositivo": "CPU, 1 hilo",
             **timings(cpu_model, G.target_layer(cpu_model, arch), x_sel[:1].cpu(), int(target_sel[0]), False)})
cost = pd.DataFrame(rows)
cost["oclusion_ventanas"] = len(range(0, run.input_size - PATCH + 1, STRIDE)) ** 2
cost.to_csv(OUT / "costo.csv", index=False)
cost.round(2)

,dispositivo,prediccion,gradcam,oclusion,oclusion_ventanas
0,cuda,7.15,11.64,23.79,49
1,"CPU, 1 hilo",8.64,8.57,221.93,49


## 9. Sensibilidad a los parámetros

Se cambia un parámetro a la vez y se mira qué tanto cambian los mapas y la prueba de fidelidad, con la
misma muestra de 504 imágenes. En Grad-CAM se prueba otra capa. En oclusión se prueba otro tamaño de
ventana (el paso es siempre la mitad de la ventana). La correlación se mide contra la configuración usada
en el resto del notebook (capa `features[12]` y ventana de 56). Una correlación cerca de 1 quiere decir que
el mapa casi no cambia.

In [11]:
LAYERS = {12: "7x7, 576 canales (la usada)", 11: "7x7, 96 canales", 8: "14x14, 48 canales", 3: "28x28, 24 canales"}
WINDOWS = [28, 56, 84]


def maps_corr(a, b):
    return float(np.nanmean([np.corrcoef(a[i].ravel(), b[i].ravel())[0, 1] if a[i].std() > 0 and b[i].std() > 0 else np.nan
                             for i in range(len(a))]))


def fidelity_row(maps, method, param, base_maps):
    f = X.deletion_fidelity(run.model, x_s, maps, target_s, FRACTION, N_RANDOM, seed=SEED)
    return {"metodo": method, "parametro": param,
            "corr_con_la_usada": maps_corr(maps, base_maps),
            "caida_top": f.caida_top.mean(), "caida_azar": f.caida_azar.mean(),
            "top_menos_azar": (f.caida_top - f.caida_azar).mean(),
            "pct_top_mayor_azar": 100 * (f.caida_top > f.caida_azar).mean(),
            "centro": float(X.center_share(maps).mean())}


rows = []
for idx, desc in LAYERS.items():
    layer = run.model.features[idx]
    local = G.GradCAM(run.model, layer)
    maps = np.concatenate([local(x_s[i:i + 64], target_s[i:i + 64])[0] for i in range(0, len(x_s), 64)])
    local.close()
    rows.append(fidelity_row(maps, "Grad-CAM", f"capa features[{idx}] ({desc})", cams_s))

for win in WINDOWS:
    if win == PATCH:
        maps = occ_s
    else:
        maps = np.stack([X.normalize_map(X.occlusion_map(run.model, x_s[i:i + 1], int(target_s[i]), win, win // 2))
                         for i in range(len(x_s))])
    rows.append(fidelity_row(maps, "Oclusión", f"ventana {win} px, paso {win // 2} px", occ_s))

sens = pd.DataFrame(rows)
sens.to_csv(OUT / "sensibilidad.csv", index=False)
sens.round(3)

,metodo,parametro,corr_con_la_usada,caida_top,caida_azar,top_menos_azar,pct_top_mayor_azar,centro
0,Grad-CAM,"capa features[12] (7x7, 576 canales (la usada))",1.000,3.966,1.007,2.959,93.056,0.310
1,Grad-CAM,"capa features[11] (7x7, 96 canales)",0.868,3.840,1.007,2.833,93.056,0.325
2,Grad-CAM,"capa features[8] (14x14, 48 canales)",0.209,4.405,1.007,3.398,85.516,0.253
3,Grad-CAM,"capa features[3] (28x28, 24 canales)",0.101,4.498,1.007,3.491,81.349,0.224
4,Oclusión,"ventana 28 px, paso 14 px",0.571,6.162,1.007,5.155,99.206,0.264
5,Oclusión,"ventana 56 px, paso 28 px",1.000,4.832,1.007,3.825,98.016,0.198
6,Oclusión,"ventana 84 px, paso 42 px",0.469,3.706,1.007,2.699,93.651,0.242


## 10. Resumen de los números

Valores que se citan en el informe, tomados de las tablas de arriba.

In [12]:
summary = {
    "errores_ocupado_libre_test": int(len(errs)),
    "errores_con_lluvia": int((errs.weather == "RAINY").sum()),
    "camaras_de_los_errores": sorted(errs.camera.unique().tolist()),
    "centro_medio_muestra_global": float(sample.centro.mean()),
    "centro_medio_errores": float(X.center_share(cams_e).mean()),
    "fidelidad_top_menos_azar": float(fid_table.loc["todas", "top_menos_azar"]),
    "fidelidad_pct_top_mayor_azar": float(fid_table.loc["todas", "pct_top_mayor_azar"]),
    "corr_gradcam_capa_11": float(sens.corr_con_la_usada.iloc[1]),
    "corr_gradcam_capa_8": float(sens.corr_con_la_usada.iloc[2]),
    "corr_gradcam_capa_3": float(sens.corr_con_la_usada.iloc[3]),
    "corr_oclusion_ventana_28": float(sens.corr_con_la_usada.iloc[4]),
    "corr_oclusion_ventana_84": float(sens.corr_con_la_usada.iloc[6]),
}
(OUT / "resumen.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
pd.Series(summary).to_frame("valor")

,valor
errores_ocupado_libre_test,10
errores_con_lluvia,10
camaras_de_los_errores,"[4, 5, 7, 8, 9]"
centro_medio_muestra_global,0.310411
centro_medio_errores,0.25392
fidelidad_top_menos_azar,2.958772
fidelidad_pct_top_mayor_azar,93.055556
corr_gradcam_capa_11,0.867736
corr_gradcam_capa_8,0.209058
corr_gradcam_capa_3,0.100742
